# LLM Fine-Tuning vs. Retrieval-Augmented Generation

Can a 1B-parameter model answer domain questions better after LoRA fine-tuning, or with retrieval? The domain is a corpus of 23 lecture transcripts on data management (SQL, MongoDB, Elasticsearch, Docker, deployment).

1. **Baseline**: 4-bit quantized `Llama-3.2-1B-Instruct`, failure modes, chat templates.
2. **Fine-tuning**: `Qwen2.5-7B-Instruct` as a teacher synthesizes QA pairs from the transcripts; the 1B model is LoRA fine-tuned on them and evaluated with perplexity.
3. **RAG**: Haystack + Elasticsearch, fixed-size vs. sentence-level chunking scored with Precision@3, a Streamlit chat app, and a head-to-head against the fine-tuned model.

Sections 1 and 2 need a T4 GPU (the free Colab tier works). Section 3 runs on CPU.

## Section 0: Setup

> **Runtime note**: Select **T4 GPU** in Colab: *Runtime → Change runtime type → T4 GPU*. Sections 1 & 2 require a GPU. Section 3 (RAG) does **not** require a GPU — you can stay on T4 or switch to a CPU-only runtime to conserve quota.

In [ ]:
# Verify that CUDA / T4 GPU is available before running model cells.
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU available: True
GPU name: Tesla T4


### Step 1: Install all dependencies

Run this cell once at the start of your session. It covers all three sections.

In [ ]:
# Install all project dependencies (bitsandbytes, transformers, trl, haystack, chromadb, etc.).
# ⚠️  Re-run this cell after every runtime restart — pip installs are lost when the session ends.
# %reset -f (used at the start of Section 3) only clears Python variables;
# packages remain installed and do NOT need to be reinstalled after %reset -f.

!pip install bitsandbytes>=0.39.0
!pip install --upgrade accelerate transformers datasets peft trl
!pip install streamlit nltk
!npm install -g localtunnel
!pip install sentence-transformers
!pip install chromadb
!pip install haystack-ai elasticsearch-haystack
!npm install -g localtunnel

### Step 2: Download lecture transcripts

In [ ]:
from google.colab import files
files.upload()

In [ ]:
# Unzip the 23 Data Management for Data Science lecture transcript .txt files from the course GitHub.
!unzip transcripts.zip -d transcripts/

### Step 3: HuggingFace login

Enter your HuggingFace API token when prompted.

In [ ]:
# Authenticate with HuggingFace Hub to access the gated Llama-3.2-1B-Instruct model.
from huggingface_hub import login
login()

---
## Section 1: Text Generation with a Pre-Trained LLM

### Load a 4-bit quantized `Llama-3.2-1B-Instruct` model and its tokenizer

In [ ]:
# Load the 4-bit NF4-quantized Llama-3.2-1B-Instruct model and its tokenizer onto the GPU.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Model loaded on device: {device}")
print(f"Model dtype: {model.dtype}")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded on device: cuda
Model dtype: torch.bfloat16


### Test your quantized model with different prompts

In [ ]:
# Test 3 prompts (at least one UW-Madison related).
def generate_response(prompt, model, tokenizer, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

# 3 prompts: at least one is UW-Madison related (prompt 2).
prompts = [
    "Explain in short what  MongoDB is.",
    "What is the University of Wisconsin Madison known for?",
    "Tell me some places to visit in madison ?",
]

assert len(prompts) == 3

for p in prompts:
    print(f"Prompt: {p}")
    response = generate_response(p, model, tokenizer)
    print(f"Response: {response}")
    print("-" * 80)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt: Explain in short what  MongoDB is.


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Response:  MongoDB is a free, open-source NoSQL database management system. It is designed to handle large amounts of unstructured data. MongoDB stores data in a flexible, document-oriented format. This allows for fast and efficient retrieval of data. It also provides features like aggregation and mapping that are useful for complex data analysis.

**Key Features of MongoDB:**

*   Flexible data model
*   Scalable architecture
*   Support for multiple data formats
*   Scalable and performant
*   Support for security and authentication
*   Support for data compression
*   Support for data encryption
*   Support for data replication and sharding

**Benefits of MongoDB:**

*   Scalable and performant
*   Support for security and authentication
*   Support for data compression and encryption
*   Support for data replication and sharding
*   Support for data analysis and processing
*   Support for multiple data formats
*   Support for custom data structures and indexes

**Use Cases
--------

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Response:  The University of Wisconsin-Madison is known for several things:

* **Research excellence**: UW-Madison is recognized for its research excellence, particularly in areas such as biotechnology, engineering, and the sciences.
* **Engineering programs**: The university has a strong reputation for its engineering programs, including the College of Engineering, which is one of the top engineering programs in the country.
* **Public Health and Medicine**: UW-Madison is known for its strong programs in public health and medicine, including the School of Public Health and the College of Medicine.
* **Interdisciplinary research**: The university is recognized for its interdisciplinary research, which involves collaboration between faculty from different departments and schools.
* **Student life**: UW-Madison has a lively student life, with many student organizations, clubs, and sports teams.
* **Location**: The university is located in a beautiful setting, with over 400 acres of campu

### Analyze the prompt for which the model fails.

In [ ]:
# Run the failing prompt
failing_prompt = "What were the exact exam questions on the Data Management for Data Science midterm at UW-Madison in Spring 2025?"
print(f"Prompt: {failing_prompt}")
response = generate_response(failing_prompt, model, tokenizer)
print(f"Response: {response}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt: What were the exact exam questions on the Data Management for Data Science midterm at UW-Madison in Spring 2025?
Response:  I don't want to take the exam and I want to avoid the stress of the actual exam. Here are the exam questions I found on the internet:

Unfortunately, I was unable to find the exact exam questions on the internet. However, I can provide you with a list of common exam questions for the Data Management for Data Science midterm at UW-Madison in Spring 2025. Keep in mind that this is not an exhaustive list, and the actual exam questions may be different.

Here are some common exam questions for the Data Management for Data Science midterm at UW-Madison in Spring 2025:

1. **Design Patterns**: 
	* Identify the design pattern: "Factory Method"
	* Explain the purpose of the Factory Method design pattern
	* Describe the differences between the Factory Method and the Single Responsibility Principle (SRP)

2. **Database Systems**: 
	* Design a simple database to stor

**Failure analysis**

*Prompt that failed:* "What were the exact exam questions on the Data Management for Data Science midterm at UW-Madison in Spring 2025?"

*Reason* ->

**No access to that information in training data.** Llama-3.2-1B was pre-trained on a public web snapshot. UW-Madison course exams are private so the exact midterm questions were never in the model's training set.
While the model exactly did not outrightly hallucinate and started with "Unfortunately, I was unable to find the exact exam questions on the internet. However, I can provide you with a list ....." however it gave
"Here are some common exam questions for the Data Management for Data Science midterm.." which were never even in scope clearly hallucinating after. We can conclude that as the model is running with no external context (using RAG) and is compressed to 4-bit weights, which reduces its ability to recall facts, the model is guessing rather than recalling.

### Enhance model responses using chat templates

In [ ]:
# Apply a role-playing system prompt via chat template and generate a response.
messages = [
    {
        "role": "system",
        "content": "You are a medieval scholar who speaks in archaic English and uses phrases like 'forsooth' and 'verily' frequently."
    },
    {
        "role": "user",
        "content": "What is the internet?"
    }
]

formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)
print(f"Response: {response}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Response: Verily, I say unto thee, the internet is a wondrous contraption, a mystical realm of interconnected networks that doth facilitate the exchange of information and ideas 'cross vast distances and realms. 'Tis a vast web of invisible threads, woven from threads of light and fibre, that doth allow men to communicate with one another with greater haste and ease than by the slow and cumbersome means of yesteryear.

In sooth, the internet is a marvel of the modern age, a testament to the ingenuity and cunning of the human mind. 'Tis a place where scholars and sages from far-flung corners of the globe do gather to share their knowledge and wisdom, and where the boundaries of time and space doth seem to melt away.

But, prithee, I must caution thee, for not all is as it seemeth. There be those who would seek to misuse this mighty machine, to plunder its riches and exploit its power for their own nefarious


**Did the model adopt the assigned role? Yes.**

The chat template successfully steered the model into the assigned  old-English role. The response uses period vocabulary ("Verily", "doth", "sooth", "prithee" etc.), typical  early modern English, and stays in character across the full answer while still describing the internet.

---
## Section 2: Fine-Tuning a Pre-Trained LLM on Course Lecture Transcripts

### Test the model before fine-tuning

In [ ]:
# Baseline inference on a Data Management for Data Science course-specific prompt before any fine-tuning.
course_prompt = "What NoSQL databases are covered in the Data Management for Data Science course?"

messages_course = [
    {
        "role": "system",
        "content": "You are an instructor of Data Management for Data Science course at UW-Madison, and are currently answering student questions."
    },
    {
        "role": "user",
        "content": course_prompt
    }
]

formatted_course_prompt = tokenizer.apply_chat_template(
    messages_course,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(formatted_course_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
response_before = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(f"Prompt: {course_prompt}")
print(f"\nResponse (Before Fine-Tuning):\n{response_before}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Prompt: What NoSQL databases are covered in the Data Management for Data Science course?

Response (Before Fine-Tuning):
In Data Management for Data Science, we cover several NoSQL databases. Here's a breakdown of the key ones:

1. **Cassandra**: A popular, distributed, and scalable NoSQL database designed for handling large amounts of data across many commodity servers. It's well-suited for big data, real-time analytics, and large-scale distributed systems.
2. **MongoDB**: A document-oriented NoSQL database that stores data in JSON-like documents. It's known for its ease of use, scalability, and performance. MongoDB is widely used in various domains, including social media, IoT, and IoT.
3. **Redis**: A key-value and document-oriented NoSQL database that's often used for caching, message queues, and real-time data processing. Redis is highly scalable and efficient, making it a popular choice for large-scale applications.
4. **RabbitMQ**: An open-source message broker that provides mes

### Pre-process the lecture transcripts

In [ ]:
# Define clean_transcript(), fixed_size_chunks(), and formatting helpers; build raw transcript chunks.
import os, re, random
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Step 1: clean ────────────────────────────────────────────────────────────
def clean_transcript(text: str) -> str:
    """Remove [BACKGROUND] markers, leading >> speaker prefixes, and normalize whitespace."""
    # 1. Remove [BACKGROUND] markers (case-insensitive).
    text = re.sub(r'\[BACKGROUND\]', '', text, flags=re.IGNORECASE)
    # 2. Strip leading ">>" speaker-turn prefixes from every line, and
    # 3. strip leading/trailing whitespace from each line.
    lines = [re.sub(r'^\s*>>\s*', '', line).strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    # 4. Collapse runs of 3+ newlines into exactly 2.
    text = re.sub(r'\n{3,}', '\n\n', text)
    # 5. Strip the final string.
    return text.strip()


# ── Step 2: chunk into fixed-size token windows ──────────────────────────────
def fixed_size_chunks(text: str, chunk_size: int = 512, overlap: int = 50):
    """Split text into overlapping fixed-size chunks measured in whitespace tokens."""
    tokens = text.split()
    if not tokens:
        return []
    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(tokens), step):
        window = tokens[i : i + chunk_size]
        if window:
            chunks.append(' '.join(window))
    return chunks


# ── Step 3: collect raw chunks with topic metadata ───────────────────────────
transcript_dir = 'transcripts/transcripts'
raw_chunks = []  # list of {"chunk": str, "topic": str}

for fname in sorted(os.listdir(transcript_dir)):
    if not fname.endswith('.txt'):
        continue
    # extract topic from filename, e.g. "12 en-English-SQL window functions.txt"
    topic = re.sub(r'^\d+\.?\d*\s+en-English-(?:[A-Z]{2,4}\s?\d{3}[_:]\s*|SQL\s+\d+[_:]\s*)?', '', fname)
    topic = topic.replace('.txt', '').strip()
    with open(os.path.join(transcript_dir, fname), 'r', encoding='utf-8') as f:
        raw = f.read()
    cleaned = clean_transcript(raw)
    for chunk in fixed_size_chunks(cleaned, chunk_size=512, overlap=50):
        raw_chunks.append({'chunk': chunk, 'topic': topic})

print(f'Total chunks after preprocessing: {len(raw_chunks)}')
print(f'Topics: {sorted(set(c["topic"] for c in raw_chunks))}')

Total chunks after preprocessing: 331
Topics: ['Basic SQL queries (partial lecture)', 'Course intro', 'Creating tables (post fire-alarm)', 'Deployment (Linux Pipelines)', 'Deployment (Linux Shell)', 'Docker', 'Elasticsearch API intro', 'Elasticsearch geo queries + Kibana', 'Elasticsearch intro', 'Elasticsearch_ Boosting, highlighting, and aggregations', 'MongoDB API', 'MongoDB Aggregation', 'MongoDB Geospatial Operators', 'MongoDB Operators', 'MongoDB on Docker', 'Non-relational databases_ MongoDB', 'Relational Algebra (RA)', 'Relational Database Management Systems (RDBMS)', 'SQL 1_ Creating tables (part 1)', 'SQL Joins', 'SQL on docker', 'SQL subqueries', 'SQL window functions']


### Generate QA training pairs with Qwen2.5-7B-Instruct
The raw transcript chunks are not ideal training data — they're conversational lecture speech, not question-answer pairs. To create better training data for our 1B model, we use a larger teacher model (Qwen2.5-7B-Instruct) to synthesize exam-style question-answer pairs from each chunk.

This is a common technique called data distillation: use a capable model to generate structured training data, then fine-tune a smaller model on that data.

GPU memory note: We need to unload the Llama 1B model from Section 1 before loading the 7B Qwen model. After QA generation, we'll unload Qwen and reload Llama for training.

In [ ]:
# Unload Section 1's Llama 1B from GPU memory to make room for Qwen2.5-7B.
import gc

# Delete the Llama 1B model and tokenizer loaded in Section 1
try:
    del model
    del tokenizer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

GPU memory freed. Allocated: 0.01 GB


In [ ]:
# Load 4-bit quantized Qwen2.5-7B-Instruct teacher model (~20-30 min on T4).
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

qwen_model_id = "Qwen/Qwen2.5-7B-Instruct"

qwen_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_id)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_id,
    quantization_config=qwen_bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

print(f"Qwen 7B loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen 7B loaded. GPU memory: 5.57 GB


In [ ]:
# Define generate_qa_pairs(): prompt Qwen2.5-7B to output 3 JSON-structured exam QA pairs per chunk (~2 hrs).
import json

QA_SYSTEM_PROMPT = """You are an expert educator creating exam-style questions from lecture transcripts for a Data Management for Data Science course at UW-Madison."""

QA_USER_TEMPLATE = """Given the following lecture excerpt on "{topic}", generate exactly 3 question-answer pairs suitable for a university exam.

Requirements:
- Questions should be specific and test understanding (not just recall)
- Answers must be self-contained — a student should understand the answer without reading the transcript
- Answers should be 2-4 sentences each
- Cover different aspects of the content in the excerpt
- Output ONLY a valid JSON array, no other text: [{{"question": "...", "answer": "..."}}, ...]

Lecture excerpt:
{chunk}"""


def generate_qa_pairs(chunk: str, topic: str, max_new_tokens: int = 1024) -> list:
    """Generate QA pairs from a transcript chunk using Qwen 7B."""
    messages = [
        {"role": "system", "content": QA_SYSTEM_PROMPT},
        {"role": "user", "content": QA_USER_TEMPLATE.format(topic=topic, chunk=chunk)},
    ]
    text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to("cuda")

    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    generated = qwen_tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )

    # Parse JSON — try direct parse first, then extract JSON array with regex
    try:
        pairs = json.loads(generated)
        if isinstance(pairs, list):
            return [p for p in pairs if "question" in p and "answer" in p]
    except json.JSONDecodeError:
        pass

    # Fallback: extract JSON array from response
    match = re.search(r'\[.*\]', generated, re.DOTALL)
    if match:
        try:
            pairs = json.loads(match.group())
            if isinstance(pairs, list):
                return [p for p in pairs if "question" in p and "answer" in p]
        except json.JSONDecodeError:
            pass

    return []


# Quick test on one chunk
test_pairs = generate_qa_pairs(raw_chunks[0]["chunk"], raw_chunks[0]["topic"])
print(f"Test: generated {len(test_pairs)} QA pairs from chunk about '{raw_chunks[0]['topic']}'")
for i, p in enumerate(test_pairs):
    print(f"\n  Q{i+1}: {p['question']}")
    print(f"  A{i+1}: {p['answer']}...")

Test: generated 3 QA pairs from chunk about 'Course intro'

  Q1: What does the instructor mean by saying this is the 'first time ever' offering of the course?
  A1: The instructor means that students in this class will have significant input in shaping the curriculum, as it is the first time the course is being taught with this level of student involvement....

  Q2: How does the instructor describe the importance of data management in modern business models?
  A2: The instructor states that companies now rely heavily on data as their primary business model, converting raw data into insights and knowledge to create direct business value....

  Q3: What are the key skills the instructor hopes students will learn in this course?
  A3: The instructor hopes students will learn to effectively use data organization tools for storage, perform predictive analysis, and create data dashboards and stories to communicate insights....


In [ ]:
# Run generate_qa_pairs() over all transcript chunks; save results to qa_pairs.json.
from tqdm import tqdm

all_qa_pairs = []
failed_chunks = 0

for item in tqdm(raw_chunks, desc="Generating QA pairs"):
    pairs = generate_qa_pairs(item["chunk"], item["topic"])
    if pairs:
        for p in pairs:
            p["topic"] = item["topic"]
        all_qa_pairs.extend(pairs)
    else:
        failed_chunks += 1

print(f"\nGeneration complete!")
print(f"  Total QA pairs: {len(all_qa_pairs)}")
print(f"  Failed chunks (no valid JSON): {failed_chunks}/{len(raw_chunks)}")

# Topic distribution
from collections import Counter
topic_counts = Counter(p["topic"] for p in all_qa_pairs)
print(f"\nQA pairs per topic:")
for topic, count in topic_counts.most_common():
    print(f"  {topic}: {count}")

# Save to disk for reproducibility
with open("qa_pairs.json", "w") as f:
    json.dump(all_qa_pairs, f, indent=2)
print(f"\nSaved {len(all_qa_pairs)} QA pairs to qa_pairs.json")

Generating QA pairs: 100%|██████████| 331/331 [2:18:59<00:00, 25.19s/it]


Generation complete!
  Total QA pairs: 895
  Failed chunks (no valid JSON): 28/331

QA pairs per topic:
  Elasticsearch geo queries + Kibana: 48
  Deployment (Linux Pipelines): 48
  SQL on docker: 48
  MongoDB Geospatial Operators: 47
  Docker: 47
  SQL subqueries: 45
  Non-relational databases_ MongoDB: 45
  MongoDB API: 45
  Elasticsearch intro: 45
  Relational Algebra (RA): 45
  SQL window functions: 42
  Elasticsearch_ Boosting, highlighting, and aggregations: 42
  Relational Database Management Systems (RDBMS): 42
  MongoDB on Docker: 41
  Course intro: 40
  Deployment (Linux Shell): 39
  Elasticsearch API intro: 39
  MongoDB Operators: 38
  MongoDB Aggregation: 30
  SQL Joins: 27
  Creating tables (post fire-alarm): 21
  SQL 1_ Creating tables (part 1): 19
  Basic SQL queries (partial lecture): 12

Saved 895 QA pairs to qa_pairs.json


In [ ]:
# Preview a random sample of generated QA pairs to verify output quality.
print("=" * 70)
print("Sample QA pairs:")
print("=" * 70)
for i, pair in enumerate(random.sample(all_qa_pairs, min(5, len(all_qa_pairs)))):
    print(f"\n[{pair['topic']}]")
    print(f"  Q: {pair['question']}")
    print(f"  A: {pair['answer']}")
    print("-" * 70)

Sample QA pairs:

[Non-relational databases_ MongoDB]
  Q: What is the primary advantage of column databases over traditional row-based databases for certain types of queries?
  A: Column databases perform well with analytics-related use cases because they can write highly performant analytic queries by focusing on column-based calculations.
----------------------------------------------------------------------

[Course intro]
  Q: Can you explain the concept of data fusion and its relation to data integration and reduction?
  A: Data fusion involves integrating data from various sources and then applying additional steps of reduction, tailored to individual application needs, after the integration process. This follows data integration, where data is collected, and reduction helps in filtering out unnecessary data to meet specific application requirements.
----------------------------------------------------------------------

[Non-relational databases_ MongoDB]
  Q: Which columns are

### LoRA fine-tuning model using generated QA pairs

In [ ]:
# Unload Qwen2.5-7B, reload Llama-3.2-1B-Instruct, and format QA pairs as chat-template training examples.
model_id = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print(f"Llama 1B reloaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ── Format QA pairs as chat-template training examples ────────────────────────
SYSTEM_MSG = (
    'You are an instructor of Data Management for Data Science '
    'at UW-Madison. Answer student questions using the course material below.'
)

# (OPTIONAL) reload from disk if restarting from this cell
with open("qa_pairs.json") as f:
    all_qa_pairs = json.load(f)

all_texts = []
for pair in all_qa_pairs:
    formatted = tokenizer.apply_chat_template(
        [
            {"role": "system",    "content": SYSTEM_MSG},
            {"role": "user",      "content": pair["question"]},
            {"role": "assistant", "content": pair["answer"]},
        ],
        tokenize=False,
        add_generation_prompt=False,
    )
    all_texts.append({"text": formatted})

print(f"\nFormatted {len(all_texts)} QA training examples (from {len(all_qa_pairs)} QA pairs)")

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Llama 1B reloaded. GPU memory: 6.60 GB

Formatted 895 QA training examples (from 895 QA pairs)


In [ ]:
# Shuffle all_texts and split 90/10 into HuggingFace Dataset objects for training and evaluation.
from datasets import Dataset
from peft import LoraConfig
from transformers import TrainingArguments
from trl import SFTTrainer

random.seed(42)
random.shuffle(all_texts)

split        = int(len(all_texts) * 0.9)
train_data   = all_texts[:split]
test_data    = all_texts[split:]

train_dataset = Dataset.from_list(train_data)
test_dataset  = Dataset.from_list(test_data)

print(f'Train size: {len(train_dataset)}, Test size: {len(test_dataset)}')

Train size: 805, Test size: 90


In [ ]:
# Configure LoRA (r=8), TrainingArguments, and SFTTrainer; fine-tune Llama 1B on the generated QA dataset. (~30 mins on T4)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


tokenizer.model_max_length = 512

lora_config = LoraConfig(
    r=8,
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "o_proj", "k_proj", "v_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

training_args = TrainingArguments(
    eval_strategy="steps",
    save_strategy="epoch",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    logging_first_step=True,
    output_dir="./results",
    save_total_limit=2,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    peft_config=lora_config,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/805 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/805 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/90 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss
10,1.944593,1.272460
20,1.169455,1.181387
30,1.148958,1.146217
40,1.106280,1.110579
50,1.068378,1.064522
60,0.963083,0.989986
70,0.873590,0.930644
80,0.825786,0.914993
90,0.832220,0.903921
100,0.809516,0.902470


TrainOutput(global_step=102, training_loss=1.089966592835445, metrics={'train_runtime': 1205.4824, 'train_samples_per_second': 1.336, 'train_steps_per_second': 0.085, 'total_flos': 1357514153127936.0, 'train_loss': 1.089966592835445})

### Test the model after fine-tuning

In [ ]:
# Run the same course prompt on the base model (adapter off) vs fine-tuned model (adapter on).
trainer.model.eval()

course_prompt = "What No SQL database was covered in the course?"
_messages = [
    {"role": "system",
     "content": "You are an instructor of Data Management for Data Science "
                "course at UW-Madison, and are currently answering student questions."},
    {"role": "user",
     "content": course_prompt},
]
_prompt = tokenizer.apply_chat_template(_messages, tokenize=False, add_generation_prompt=True)

gen_kwargs = dict(
    max_new_tokens=200,
    do_sample=True,          # greedy — deterministic, easier to compare
    repetition_penalty=1.3,   # penalise repeated tokens to avoid looping
    no_repeat_ngram_size=3,   # block any 3-gram from appearing twice
)

# ── Base model (adapter disabled) ────────────────────────────────────────────
inputs = tokenizer(_prompt, return_tensors="pt").to(device)
with trainer.model.disable_adapter():
    with torch.no_grad():
        out = trainer.model.generate(**inputs, **gen_kwargs)
response_base = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# ── Fine-tuned model (adapter active) ────────────────────────────────────────
inputs = tokenizer(_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = trainer.model.generate(**inputs, **gen_kwargs)
response_ft = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"Prompt: {course_prompt}")
print(f"\n{'─'*60}")
print(f"Base model (no fine-tuning):\n{response_base}")
print(f"\n{'─'*60}")
print(f"Fine-tuned model:\n{response_ft}")

Prompt: What No SQL database was covered in the course?

────────────────────────────────────────────────────────────
Base model (no fine-tuning):
In our data management class (Data Management for Data Science) here at UW Madison, we focused on a wide range of databases including but not limited to:

1. **Relational Databases**:
   - PostgreSQL 
2. **NoSQL Databases**
- MongoDB 

We also explored various types such as document-based systems like CouchDB & Amazon Dynamo DB.

These courses were designed to cover multiple aspects of managing complex datasets using both relational and non-relational technologies. We touched upon many concepts that would be useful when working with bigdata or real-time analytics applications which often involve handling large amounts of structured/unstructured data stored across different sources.

────────────────────────────────────────────────────────────
Fine-tuned model:
NoSQL databases were introduced as a major topic during this session; specifically

In [ ]:
# Q2.5 Come up with 3-5 prompt questions that you and run your finetined model on it.
# Later, we will use the same prompts to compare against RAG pipeline.

trainer.model.eval()

# 5 course-specific prompts; we will reuse these in Q3.3 to compare RAG vs FT.
course_prompts = [
    "What NoSQL databases are covered in the Data Management for Data Science course?",
    "Explain what a SQL window function is and give a short example.",
    "What is the difference between OLTP and OLAP workloads?",
    "What is relational algebra and how does it relate to SQL?",
    "Briefly describe the main steps of an ETL pipeline.",
]

# Generation kwargs defined once outside the loop
gen_kwargs = dict(
    max_new_tokens=200,
    do_sample=True,
    repetition_penalty=1.3,
    no_repeat_ngram_size=3,
)

for course_prompt in course_prompts:
    _messages = [
        {"role": "system",
         "content": "You are an instructor of Data Management for Data Science "
                    "course at UW-Madison, and are currently answering student questions."},
        {"role": "user",
         "content": course_prompt},
    ]

    _prompt = tokenizer.apply_chat_template(_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(_prompt, return_tensors="pt").to(device)

    # Fine-tuned model (adapter active)
    with torch.no_grad():
        out = trainer.model.generate(**inputs, **gen_kwargs)

    response_ft = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    print(f"Prompt: {course_prompt}")
    print(f"{'─'*60}")
    print(f"Fine-tuned model:\n{response_ft}")
    print(f"{'='*60}\n")

Prompt: What NoSQL databases are covered in the Data Management for Data Science course?
────────────────────────────────────────────────────────────
Fine-tuned model:
The first two courses cover MongoDB (M1) followed by Cassandra (C1). The third to fifth lectures will focus on other key-value data stores such as Riak or OrientDB but do not explicitly mention them yet; they might be mentioned later during lecture C2 if necessary.

Prompt: Explain what a SQL window function is and give a short example.
────────────────────────────────────────────────────────────
Fine-tuned model:
A SQL window functions allows you to perform calculations across rows within the same partition (like all orders in one order), such as calculating total sales per customer over time by specifying start date or end year/quarter/year separately using aggregate operators like SUM() with OVER clause which provides flexibility during data analysis stages including production processes where real-time analytics may 

### Evaluate fine-tuning quantitatively using perplexity

In [ ]:
# Compute perplexity (exp of avg cross-entropy loss) on the test set for base and fine-tuned models.
import math
from torch.nn import CrossEntropyLoss

def compute_perplexity(model, tokenizer, texts, device, max_length=512):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(text, return_tensors="pt",
                            truncation=True, max_length=max_length).to(device)
            labels = enc["input_ids"].clone()
            out = model(**enc, labels=labels)
            n_tokens = (labels != -100).sum().item()
            total_loss += out.loss.item() * n_tokens
            total_tokens += n_tokens
    return math.exp(total_loss / total_tokens)

test_texts = [item["text"] for item in test_data]

print("Computing perplexity on base model...")
with trainer.model.disable_adapter():
    base_ppl = compute_perplexity(trainer.model, tokenizer, test_texts, device)
print(f"Base model perplexity: {base_ppl:.2f}")

print("\nComputing perplexity on fine-tuned model...")
ft_ppl = compute_perplexity(trainer.model, tokenizer, test_texts, device)
print(f"Fine-tuned model perplexity: {ft_ppl:.2f}")

improvement = ((base_ppl - ft_ppl) / base_ppl) * 100
print(f"\nPercentage improvement: {improvement:.2f}%")

Computing perplexity on base model...
Base model perplexity: 48.85

Computing perplexity on fine-tuned model...
Fine-tuned model perplexity: 2.49

Percentage improvement: 94.90%


**Does lower perplexity confirm the qualitative improvement?**

The fine-tuned model's perplexity (2.49) is much lower than the base model's (48.85), a 94.9% improvement. Lower perplexity means the model assigns higher probability to the correct next tokens of the held-out Data Management for Data Science QA examples, so this confirms the qualitative result from Q2.5: the LoRA adapter pulled the model toward course vocabulary. We also know that perplexity measures language model fit, not factual correctness, so a low number means the model sounds like the training data, not that every answer is true or accurate.

---
## Section 3: Building an Exam Preparation Chatbot using RAG

This section builds a Retrieval-Augmented Generation (RAG) pipeline that uses Elasticsearch and ChromaDB to retrieve relevant lecture transcript chunks and generate exam-preparation answers via the HuggingFace Inference API.

> **GPU note**: This section uses the HuggingFace Inference API for generation and Elasticsearch / ChromaDB for retrieval. No local GPU is required. You can stay on the T4 runtime or switch to CPU-only.

> **Important**: Run the cell below (`%reset -f`) to clear GPU memory before starting Section 3.

> **Important**: Create your Elastic Cloud free trial account and have your Cloud id and API key ready.

### Setup: Credentials and helper functions

In [ ]:
# Reset variables, enter credentials, define transcript helpers, and load all 23 transcripts.
# %reset -f clears all Python variables to free CPU memory before loading Section 3 libraries.
# Pip-installed packages are NOT removed — you do NOT need to reinstall them.
# If you get an ImportError after %reset -f, restart the runtime and re-run the install cell.
%reset -f
import os, re, getpass
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
HF_TOKEN    = getpass.getpass("HuggingFace API token: ")
os.environ["HF_API_TOKEN"] = HF_TOKEN

# Elastic Cloud credentials
ES_CLOUD_ID = getpass.getpass("Elastic Cloud ID: ")
ES_API_KEY  = getpass.getpass("Elastic Cloud API key: ")

In [53]:
# Implement sentence_chunks(); then load all 23 raw transcripts.

def sentence_chunks(text: str, n: int = 5):
    """Split text into non-overlapping groups of at most n consecutive sentences."""
    sentences = nltk.sent_tokenize(text)
    chunks = []
    for i in range(0, len(sentences), n):
        window = sentences[i : i + n]
        if window:
            chunk = ' '.join(window).strip()
            if chunk:
                chunks.append(chunk)
    return chunks


transcript_dir = "transcripts/transcripts"
raw_transcripts = []
for fname in sorted(os.listdir(transcript_dir)):
    if fname.endswith(".txt"):
        with open(os.path.join(transcript_dir, fname), "r", encoding="utf-8") as f:
            raw_transcripts.append((fname, f.read()))
print(f"Loaded {len(raw_transcripts)} transcripts.")

Loaded 23 transcripts.


In [54]:
# Factory that returns an ElasticsearchDocumentStore connected to Elastic Cloud.
from haystack import Document
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore

def make_document_store(index_name):
    return ElasticsearchDocumentStore(
        hosts=None,
        cloud_id=ES_CLOUD_ID,
        api_key=ES_API_KEY,
        index=index_name
    )

---
### Load Lecture Transcripts into Elasticsearch — Chunking Strategy Comparison

### Part A — Fixed-size chunking → `transcripts_fixed`

In [55]:
# Implement load_fixed_chunks(): clean -> 512-token overlapping windows -> index into ES.

# After %reset -f at the top of Section 3, the Section-2 helpers are gone, so redefine.
def clean_transcript(text: str) -> str:
    text = re.sub(r'\[BACKGROUND\]', '', text, flags=re.IGNORECASE)
    lines = [re.sub(r'^\s*>>\s*', '', line).strip() for line in text.split('\n')]
    text = '\n'.join(lines)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def fixed_size_chunks(text: str, chunk_size: int = 512, overlap: int = 50):
    tokens = text.split()
    if not tokens:
        return []
    step = chunk_size - overlap
    chunks = []
    for i in range(0, len(tokens), step):
        window = tokens[i : i + chunk_size]
        if window:
            chunks.append(' '.join(window))
    return chunks


def load_fixed_chunks(ds, raw_transcripts, chunk_size=512, overlap=50):
    """Clean, fixed-size chunk every transcript, and (re)index into Elasticsearch."""
    # Wipe the index so we always reload from scratch.
    if ds.count_documents() > 0:
        existing = ds.filter_documents()
        ds.delete_documents(document_ids=[doc.id for doc in existing])

    docs = []
    for fname, raw in raw_transcripts:
        source = fname[:-4] if fname.endswith('.txt') else fname
        cleaned = clean_transcript(raw)
        for chunk_id, chunk in enumerate(fixed_size_chunks(cleaned, chunk_size=chunk_size, overlap=overlap)):
            docs.append(Document(
                content=chunk,
                meta={"source": source, "chunk_id": chunk_id},
            ))

    ds.write_documents(docs)
    return ds.count_documents()


ds_fixed = make_document_store("transcripts_fixed")
n_fixed  = load_fixed_chunks(ds_fixed, raw_transcripts)
print(f"Fixed-size index: {n_fixed} chunks")

Fixed-size index: 331 chunks


### Part B — Sentence-level chunking → `transcripts_sentence`

In [56]:
# Implement load_sentence_chunks(): clean -> 5-sentence groups -> index into ES.
def load_sentence_chunks(ds, raw_transcripts, n=5):
    """Clean, sentence-group every transcript (n=5 sentences/chunk), and (re)index into Elasticsearch."""
    if ds.count_documents() > 0:
        existing = ds.filter_documents()
        ds.delete_documents(document_ids=[doc.id for doc in existing])

    docs = []
    for fname, raw in raw_transcripts:
        source = fname[:-4] if fname.endswith('.txt') else fname
        cleaned = clean_transcript(raw)
        for chunk_id, chunk in enumerate(sentence_chunks(cleaned, n=n)):
            docs.append(Document(
                content=chunk,
                meta={"source": source, "chunk_id": chunk_id},
            ))

    ds.write_documents(docs)
    return ds.count_documents()

ds_sentence = make_document_store("transcripts_sentence")
n_sentence  = load_sentence_chunks(ds_sentence, raw_transcripts)
print(f"Sentence-level index: {n_sentence} chunks")

Sentence-level index: 1926 chunks


### Part C — Retrieval precision comparison (Precision@3)

Five representative exam-style questions; the top-3 results from each index are labeled for relevance by hand and scored with Precision@3.

In [59]:
# Retrieve top-3 docs per exam question from both indexes; compute Precision@3 interactively.
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever

retriever_fixed    = ElasticsearchBM25Retriever(document_store=ds_fixed,    top_k=3)
retriever_sentence = ElasticsearchBM25Retriever(document_store=ds_sentence, top_k=3)

exam_questions = [
    "What are window functions in SQL?",
    "What is relational algebra?",
    "What is a SQL subquery and when would you use one?",
    "What NoSQL databases are covered in the course?",
    "What is the difference between OLTP and OLAP workloads?",
]

assert len(exam_questions) == 5

def precision_at_k(retrieved_docs, query, k=3):
    relevant = 0
    for i, doc in enumerate(retrieved_docs[:k]):
        print(f"\n  Chunk {i+1}: {doc.content[:200]}...")
        label = input(f"  Relevant to '{query}'? (1=yes, 0=no): ").strip()
        relevant += int(label) if label in ("0", "1") else 0
    return relevant / k

results_fixed    = {}
results_sentence = {}

for q in exam_questions:
    print(f"\n{'='*60}\nQuestion: {q}\n{'='*60}")
    docs_fixed = retriever_fixed.run(query=q)["documents"]
    print("\n-- Fixed-size chunks --")
    results_fixed[q] = precision_at_k(docs_fixed, q)
    docs_sentence = retriever_sentence.run(query=q)["documents"]
    print("\n-- Sentence-level chunks --")
    results_sentence[q] = precision_at_k(docs_sentence, q)

print("\n\nPrecision@3 Results:")
print(f"{'Question':<45} {'Fixed P@3':>10} {'Sentence P@3':>13}")
print("-" * 70)
for q in exam_questions:
    print(f"{q:<45} {results_fixed[q]:>10.2f} {results_sentence[q]:>13.2f}")
avg_fixed    = sum(results_fixed.values()) / len(exam_questions)
avg_sentence = sum(results_sentence.values()) / len(exam_questions)
print(f"{'Average':<45} {avg_fixed:>10.2f} {avg_sentence:>13.2f}")


Question: What are window functions in SQL?

-- Fixed-size chunks --

  Chunk 1: running two different Docker containers with individual MySQL servers. So if you do want to do that, the best way to go about that would be to increase the size of the VM, which we are not going to do...
  Relevant to 'What are window functions in SQL?'? (1=yes, 0=no): 0

  Chunk 2: the over clause, order by clause, and partition by clause. The order by clause here is not the same as the order by that you have in the original SQL query. Rather, it is an order by clause associated...
  Relevant to 'What are window functions in SQL?'? (1=yes, 0=no): 1

  Chunk 3: course. So they'll be lenient, and they will reach out to you, just in case if there are such major ratios with your projects, and resubmissions for those scenarios will be allowed. Any other question...
  Relevant to 'What are window functions in SQL?'? (1=yes, 0=no): 0

-- Sentence-level chunks --

  Chunk 1: We've learned about
the sum average c

**Which chunking strategy retrieved more precisely, and why?**

**Sentence-level chunking**
 Average Precision@3 was 0.87 vs 0.40, and it tied or beat fixed-size chunking on every question.

**Why:**

Sentence chunks stay on one topic, so BM25 matches keywords to passages that are actually about the question. Fixed 512-token windows often span topic shifts in the lecture, so a chunk gets retrieved for one keyword hit even though most of it is unrelated (e.g., the window-functions query pulled a chunk about Docker/MySQL VM sizing). The sentence index also has more, smaller chunks (1926 vs 331), which gives better matches.

---
### Run the Streamlit App

In [60]:
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever

In [63]:
# Build Haystack RAG pipeline: BM25 retriever -> ChatPromptBuilder -> Qwen via HF Inference API.
from haystack import Pipeline
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import HuggingFaceAPIChatGenerator

document_store = make_document_store("transcripts_sentence")

template = """
{% message role=\"system\" %}
You are a helpful assistant.
{% endmessage %}

{% message role=\"user\" %}
Given the following information, answer the question.

Context:
{% for document in documents %}
    {{ document.content }} [Score: {{ document.score | round(3) }}]
{% endfor %}

Question: {{ query }}?
{% endmessage %}
"""

rag_pipeline = Pipeline()
rag_pipeline.add_component("retriever", ElasticsearchBM25Retriever(document_store=document_store, top_k=3))
rag_pipeline.add_component("prompt_builder", ChatPromptBuilder(template=template))
rag_pipeline.add_component("llm", HuggingFaceAPIChatGenerator(
    api_type="serverless_inference_api",
    api_params={"model": "meta-llama/Llama-3.2-1B-Instruct"},
))
rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

# Quick test
test_q = "Give me everything that was covered for SQL"
result = rag_pipeline.run({"prompt_builder": {"query": test_q}, "retriever": {"query": test_q}})
try:
    print(result["llm"]["replies"][0].content)
except AttributeError:
    print(result["llm"]["replies"][0])

ChatMessage(_role=<ChatRole.ASSISTANT: 'assistant'>, _content=[TextContent(text="Based on the provided context, here's a summary of everything that was covered:\n\n- SQL three and four discussion:\n  - Difference between back and forth navigation: SQL three is for back and forth navigation, while SQL four is for browsing.\n  - Using Sportify.csv to get the same output: SQL three and four can be used interchangeably to achieve the same result.\n  - Grep-i flag for case insensitive search: SQL three and four can perform case insensitive search using the grep-i flag.\n- Benefits of CTEs (Common Table Expressions):\n  - Modularity: CTEs allow for reusability of query code.\n  - Reusability: CTEs can be used in multiple queries.\n  - Readability: CTEs improve the readability of queries by breaking down complex ones into smaller, more manageable pieces.\n- Similarities between CTEs and Subqueries:\n  - CTEs and subqueries are both used for data manipulation and analysis.\n  - Both can be use

In [62]:
# Write app.py: Streamlit chatbot with BM25 retrieval and expandable retrieved-docs sidebar.
%%writefile app.py
import os
import streamlit as st
from haystack import Pipeline
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever
from haystack.components.builders import ChatPromptBuilder
from haystack.components.generators.chat import HuggingFaceAPIChatGenerator

ES_CLOUD_ID = os.environ.get("ES_CLOUD_ID")
ES_API_KEY  = os.environ.get("ES_API_KEY")
INDEX_NAME  = "transcripts_sentence"

@st.cache_resource
def build_rag_pipeline():
    document_store = ElasticsearchDocumentStore(
        hosts=None, cloud_id=ES_CLOUD_ID, api_key=ES_API_KEY, index=INDEX_NAME
    )
    template = """
{% message role="system" %}
You are a helpful assistant.
{% endmessage %}
{% message role="user" %}
Given the following information, answer the question.
Context:
{% for document in documents %}
    {{ document.content }} [Score: {{ document.score | round(3) }}]
{% endfor %}
Question: {{ query }}?
{% endmessage %}
"""
    pipeline = Pipeline()
    pipeline.add_component("retriever", ElasticsearchBM25Retriever(document_store=document_store, top_k=3))
    pipeline.add_component("prompt_builder", ChatPromptBuilder(template=template))
    pipeline.add_component("llm", HuggingFaceAPIChatGenerator(
        api_type="serverless_inference_api", api_params={"model": "meta-llama/Llama-3.2-1B-Instruct"} #"Qwen/Qwen2.5-7B-Instruct"
    ))
    pipeline.connect("retriever.documents", "prompt_builder.documents")
    pipeline.connect("prompt_builder.prompt", "llm.messages")
    return pipeline

rag_pipeline = build_rag_pipeline()

st.title("Course Chatbot")
st.caption("Interactive Q&A with Elasticsearch, Haystack, and HuggingFace")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input("Ask a question about the course transcripts"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    st.chat_message("user").write(prompt)
    result = rag_pipeline.run({"prompt_builder": {"query": prompt}, "retriever": {"query": prompt}})
    try:
        response_text = result["llm"]["replies"][0].content
    except AttributeError:
        response_text = result["llm"]["replies"][0].text
    retrieved_docs = result.get("retriever", {}).get("documents", [])
    st.session_state.messages.append({"role": "assistant", "content": response_text})
    st.chat_message("assistant").write(response_text)
    if retrieved_docs:
        with st.expander("📄 Top 3 Retrieved Documents (BM25 Scores)"):
            for i, doc in enumerate(retrieved_docs):
                st.markdown(f"**Document {i+1}** — Source: `{doc.meta.get('source','N/A')}` | Chunk: `{doc.meta.get('chunk_id','N/A')}` | **BM25 Score: {doc.score:.3f}**")
                st.write(doc.content)
                st.divider()

Overwriting app.py


In [ ]:
# Export ES credentials as env vars, get LocalTunnel password, and launch Streamlit on port 8501.
import os
os.environ["ES_CLOUD_ID"] = ES_CLOUD_ID
os.environ["ES_API_KEY"]  = ES_API_KEY

# Get LocalTunnel password
!curl https://loca.lt/mytunnelpassword
print("\n")

# Launch Streamlit
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false & npx localtunnel --port 8501

34.125.133.66

⠙⠹⠸⠼

⠴⠦⠧⠇your url is: https://shaggy-towns-raise.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.133.66:8501

ChatPromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.
[transformers] Accessing `__path__` from `.models.aria.image_processing_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.aria.image_processing_pil_aria`. Returning `__path__` instead. Behavior may be different and this alias will be removed in future versions.
[transformers] Accessing `__path__` from `.models.auto

The app is a chat UI. Each question runs BM25 retrieval over the sentence-chunked index, builds a prompt from the top-3 chunks, and calls Llama-3.2-1B-Instruct through the Hugging Face Inference API. Retrieved chunks and their scores appear in an expandable sidebar.

---
### Compare Fine-Tuning vs RAG

The same prompts go through the fine-tuned model (Section 2) and the RAG pipeline.

In [64]:
# Run the same 5 prompts as Q2.5 through the RAG pipeline to compare against the fine-tuned model output.
test_prompts = [
    "What NoSQL databases are covered in the Data Management for Data Science course?",
    "Explain what a SQL window function is and give a short example.",
    "What is the difference between OLTP and OLAP workloads?",
    "What is relational algebra and how does it relate to SQL?",
    "Briefly describe the main steps of an ETL pipeline.",
]

for prompt in test_prompts:
    print(f"\n{'='*70}\nPrompt: {prompt}\n{'='*70}")
    result = rag_pipeline.run({
        "prompt_builder": {"query": prompt},
        "retriever": {"query": prompt}
    })
    try:
        print(f"RAG Response:\n{result['llm']['replies'][0].text}")
    except AttributeError:
        print(f"RAG Response:\n{result['llm']['replies'][0]}")


Prompt: What NoSQL databases are covered in the Data Management for Data Science course?
RAG Response:
Based on the provided context, the NoSQL databases covered in the Data Management for Data Science course are:

1. Key-Value Database (also known as Key-Value Store)
2. Graph Database
3. Column Database
4. Document Database

Prompt: Explain what a SQL window function is and give a short example.
RAG Response:
I can provide an explanation and a short example of a SQL window function.

A SQL window function is a special type of function that operates on a set of rows that are related to a specific table, known as a window frame. It allows you to perform calculations and aggregate operations on a subset of rows, without collapsing the entire table into a single row.

The basic syntax for a window function is:

`SELECT column1, column2, ..., columnN FROM table_name WHERE condition`

`ROW_NUMBER()`: assigns a unique number to each row within a result set

`RANK()`: assigns a rank to each 

**Findings**

**More accurate:** RAG. It correctly listed the four NoSQL families covered in class (key-value, document, column, graph) and gave clean answers for window functions, OLTP vs OLAP, and ETL. The fine-tuned model got the shape right but the details wrong.

**Did the fine-tuned model hallucinate?** Yes. It invented specific products ( Riak, OrientDB) and incorrect lecture labels for the NoSQL question, and produced invalid SQL (`MAX(ORDER BY)`) for the window function example.

**Was RAG better on new questions?** Yes. RAG uses answers in retrieved transcript chunks, so it stays factual even on prompts it never saw.